# COM7021 Data Visualisation — EcoMoveMobility Investigation

**Assessment:** COM7021 Assessment  
**Dataset:** `[5442] EcoMoveMobility.xlsx`  
**Purpose:** Explore sustainable urban mobility performance across dates, cities, countries, transport modes, route types, weather and event days. This notebook contains reproducible Python analysis, ten distinct visualisations, evidence-led interpretations, and the source code for an interactive Streamlit dashboard.

**Programme:** MSc Data Science

> **Submission note:** This notebook retains the original investigation and uses figure numbers matching the accompanying Word report (Figures 1–10). The Word appendix must also include the complete code of the Streamlit generator in cell 32. Submit files according to the module guidance.

**Dataset-based revision:** The uploaded source file, `[5442] EcoMoveMobility.xlsx` (sheet `Dataset`), is used by this notebook. The report-aligned Figures 1–10, accessible blue scales, the delay–rating correlation and the complete dashboard-source generator are retained. The final cell audits exports and reconciles key results with the supplied Word report. Run all cells after moving the notebook and Excel workbook together.


## 1. What this notebook does

This investigation is structured to address the practical requirements for **Task 1 (LO1 and LO2)** and provide evidence for **Task 2 (LO3 and LO4)**.

- Validate the data structure and check missing values, duplicate rows, duplicate record IDs and date coverage.
- Create ten visualisations in the **same figure order as the Word report**: demand by month and city (1–2), on-time by country and mode/route (3–4), delay/rating and weather (5–6), event-day performance (7), finance (8), access (9), and CO₂ (10).
- Add a brief explanation of why each visualisation is suitable, what it shows, and how managers could use it.
- Export each interactive chart as an HTML file to `ecomove_outputs/`.
- Create a standalone interactive dashboard source file named `ecomove_dashboard.py` with date/category filters and dynamically recalculated KPIs and charts.

### Software

Run the notebook in Jupyter Notebook, JupyterLab, VS Code, or Google Colab. Required libraries are `pandas`, `numpy`, `plotly` and `openpyxl`. To run the separate dashboard app, also install `streamlit`.

If a package is missing, run this in a separate setup cell: `pip install pandas numpy plotly openpyxl streamlit`

In [35]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
px.defaults.template = "plotly_white"

# Use the original dataset filename.
DATA_FILENAMES = [
    "[5442] EcoMoveMobility.xlsx"
]
OUTPUT_DIR = Path("ecomove_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Look for the exact uploaded file alongside the notebook, or in /mnt/data.
search_dirs = [Path.cwd(), Path("/mnt/data")]
candidates = [folder / filename for folder in search_dirs for filename in DATA_FILENAMES]
DATA_PATH = next((path for path in candidates if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Cannot locate the EcoMoveMobility workbook. Place "
        "'[5442] EcoMoveMobility.xlsx' "
        "in the notebook folder and rerun the notebook."
    )

print(f"Python: {sys.version.split()[0]}")
print(f"Dataset: {DATA_PATH.resolve()}")
print(f"Chart exports: {(Path.cwd() / OUTPUT_DIR).resolve()}")

Python: 3.13.18
Dataset: /content/[5442] EcoMoveMobility.xlsx
Chart exports: /content/ecomove_outputs


## 2. Load and prepare the dataset

The source workbook includes a `Dataset` worksheet. Excel stores dates as serial numbers in this workbook, so the `Date` column is explicitly converted using Excel's 1900 date system. Numeric variables are converted to numeric data types before aggregation. A copy of the original values is not needed for this descriptive analysis; the original workbook remains unchanged.

In [36]:
# Load the provided sheet
raw_df = pd.read_excel(DATA_PATH, sheet_name="Dataset")
df = raw_df.copy()

# Convert Excel serial dates correctly. If the field is already text/date-like,
# use normal date parsing instead.
if pd.api.types.is_numeric_dtype(df["Date"]):
    df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30", errors="coerce")
else:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_cols = [
    "Passengers", "Trips", "Average_Delay_Minutes", "On_Time_Percentage",
    "Ticket_Revenue_EUR", "Operating_Cost_EUR", "CO2_Saved_KG",
    "Energy_Consumption_KWh", "Customer_Rating", "Accessibility_Complaints"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")
print(f"Date range: {df['Date'].min():%d %B %Y} to {df['Date'].max():%d %B %Y}")
display(df.head(8))

Loaded 5,000 rows and 18 columns.
Date range: 01 January 2024 to 31 December 2025


,Record_ID,Date,City,Country,Transport_Mode,Route_Type,Weather,Event_Day,Passengers,Trips,Average_Delay_Minutes,On_Time_Percentage,Ticket_Revenue_EUR,Operating_Cost_EUR,CO2_Saved_KG,Energy_Consumption_KWh,Customer_Rating,Accessibility_Complaints
0,EM-00001,2025-03-31,Paris,France,Tram,Tourist Zone,Cloudy,Yes,22946,101,3.2,85.1,58846.34,47796.73,2005.0,1446.4,4.4,26
1,EM-00002,2025-03-01,Paris,France,Tram,University Area,Cloudy,No,9417,59,1.5,90.1,24578.85,20708.01,824.3,495.1,4.7,7
2,EM-00003,2025-08-13,Rotterdam,Netherlands,Tram,Airport Connector,Clear,No,6467,38,3.4,87.9,15168.68,14129.41,693.1,405.8,4.4,7
3,EM-00004,2024-07-31,Lyon,France,Tram,University Area,Clear,Yes,5218,33,6.9,74.5,14172.20,11945.52,600.1,278.1,3.9,7
4,EM-00005,2024-10-13,Amsterdam,Netherlands,Metro,Tourist Zone,Cloudy,No,10258,19,3.6,83.2,27534.92,28846.82,1392.4,580.5,4.4,12
5,EM-00006,2025-03-13,Vienna,Austria,E-Scooter,Commuter Corridor,Rain,No,394,394,4.7,84.1,563.40,10583.52,59.1,8.2,3.8,0
6,EM-00007,2024-10-16,Rotterdam,Netherlands,Park and Ride,Commuter Corridor,Clear,No,4540,35,3.5,82.9,14235.98,10442.58,281.9,154.3,4.1,8
7,EM-00008,2025-06-07,Paris,France,Bike Share,Suburban Feeder,Heatwave,No,349,349,2.1,88.2,366.88,3157.01,62.1,0.0,3.8,1


## 3. Data quality and coverage checks

These checks assess whether the supplied file is structurally ready for analysis. They do not prove that the source data is objectively accurate or representative. In particular, a complete dataset can still contain unusual values or measurement limitations.

In [37]:
required_columns = [
    "Record_ID", "Date", "City", "Country", "Transport_Mode", "Route_Type", "Weather", "Event_Day",
    "Passengers", "Trips", "Average_Delay_Minutes", "On_Time_Percentage", "Ticket_Revenue_EUR",
    "Operating_Cost_EUR", "CO2_Saved_KG", "Energy_Consumption_KWh", "Customer_Rating", "Accessibility_Complaints"
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Required columns missing from workbook: {missing_columns}")

quality_checks = pd.DataFrame({
    "Check": [
        "Rows", "Columns", "Missing cells (all columns)", "Duplicate rows",
        "Duplicate Record_ID values", "Invalid/unparsed dates", "Distinct cities",
        "Distinct countries", "Distinct transport modes", "Distinct route types",
        "Distinct weather categories", "Distinct event-day categories"
    ],
    "Result": [
        len(df), df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum()),
        int(df["Record_ID"].duplicated().sum()), int(df["Date"].isna().sum()),
        int(df["City"].nunique()), int(df["Country"].nunique()), int(df["Transport_Mode"].nunique()),
        int(df["Route_Type"].nunique()), int(df["Weather"].nunique()), int(df["Event_Day"].nunique())
    ]
})
display(quality_checks)

print("Numeric data type and range review:")
display(df[numeric_cols].describe().T.round(2))

print("Interpretation: missing values and duplicate checks are important, but they cannot verify whether a recorded delay, passenger count, cost or CO₂ figure is correct. Extreme values should be assessed in context rather than removed automatically.")

,Check,Result
0,Rows,5000
1,Columns,18
2,Missing cells (all columns),0
3,Duplicate rows,0
4,Duplicate Record_ID values,0
5,Invalid/unparsed dates,0
6,Distinct cities,12
7,Distinct countries,6
8,Distinct transport modes,6
9,Distinct route types,6


Numeric data type and range review:


,count,mean,std,min,25%,50%,75%,max
Passengers,5000.0,8181.94,10018.72,64.00,1411.75,4398.00,10077.50,66416.00
Trips,5000.0,121.77,155.79,11.00,36.00,52.00,106.00,1048.00
Average_Delay_Minutes,5000.0,4.35,2.39,0.00,2.60,4.40,6.00,12.50
On_Time_Percentage,5000.0,82.06,7.93,55.00,76.50,82.00,87.90,99.00
Ticket_Revenue_EUR,5000.0,22328.12,28822.41,90.93,3280.33,11190.02,26329.65,204386.05
Operating_Cost_EUR,5000.0,22422.00,26114.32,978.37,7189.86,11638.28,22732.98,178936.39
CO2_Saved_KG,5000.0,896.39,1261.16,8.30,126.85,338.30,1009.48,8884.10
Energy_Consumption_KWh,5000.0,466.81,564.22,0.00,85.20,269.70,599.62,3856.30
Customer_Rating,5000.0,4.01,0.40,2.70,3.70,4.00,4.30,4.90
Accessibility_Complaints,5000.0,12.53,16.97,0.00,2.00,7.00,15.00,147.00


Interpretation: missing values and duplicate checks are important, but they cannot verify whether a recorded delay, passenger count, cost or CO₂ figure is correct. Extreme values should be assessed in context rather than removed automatically.


## 4. Overall performance indicators

The following summary establishes the baseline for later comparisons. Totals reflect sums of the supplied record-level values. Average percentages, delays and ratings are arithmetic means across records; they are not passenger-weighted averages. Revenue minus operating cost is a simple comparison of the two supplied columns and should not be described as full accounting profit.

In [38]:
overall = {
    "Records analysed": len(df),
    "Total passengers (sum of record values)": df["Passengers"].sum(),
    "Total trips (sum of record values)": df["Trips"].sum(),
    "Average on-time performance (%)": df["On_Time_Percentage"].mean(),
    "Average delay (minutes)": df["Average_Delay_Minutes"].mean(),
    "Average customer rating (out of 5)": df["Customer_Rating"].mean(),
    "Total ticket revenue (€)": df["Ticket_Revenue_EUR"].sum(),
    "Total operating cost (€)": df["Operating_Cost_EUR"].sum(),
    "Revenue minus operating cost (€)": df["Ticket_Revenue_EUR"].sum() - df["Operating_Cost_EUR"].sum(),
    "Total CO₂ saved (kg)": df["CO2_Saved_KG"].sum(),
    "Total accessibility complaints": df["Accessibility_Complaints"].sum(),
}
summary_table = pd.DataFrame({"Indicator": overall.keys(), "Value": overall.values()})
display(summary_table.style.format({"Value": lambda x: f"{x:,.2f}" if isinstance(x, (int, float, np.number)) else str(x)}))

# Reusable helper: display interactive Plotly charts in the notebook and save
# a standalone HTML version which can be opened in a browser.
def show_and_save(fig, filename):
    target = OUTPUT_DIR / filename
    fig.write_html(target, include_plotlyjs="cdn", full_html=True)
    display(fig)
    print(f"Interactive chart saved to: {target.resolve()}")

,Indicator,Value
0,Records analysed,"5,000.00"
1,Total passengers (sum of record values),"40,909,721.00"
2,Total trips (sum of record values),"608,862.00"
3,Average on-time performance (%),82.06
4,Average delay (minutes),4.35
5,Average customer rating (out of 5),4.01
6,Total ticket revenue (€),"111,640,614.26"
7,Total operating cost (€),"112,110,015.97"
8,Revenue minus operating cost (€),"-469,401.71"
9,Total CO₂ saved (kg),"4,481,974.30"


## 5. Visualisation 1 — Monthly passenger demand

**Question:** How does passenger volume change across the assessment period?  
**Why this chart:** A line chart is suited to ordered time data and helps viewers identify peaks, troughs and broad changes across months.  
**Decision use:** Managers can use demand patterns to investigate staffing, capacity and service scheduling. Monthly aggregation reduces daily noise, but it can conceal important within-month peaks and does not establish why demand changed.

In [39]:
monthly_passengers = (df.dropna(subset=["Date"]).set_index("Date").resample("MS")["Passengers"].sum().reset_index())
fig1 = px.line(monthly_passengers, x="Date", y="Passengers", markers=True,
               title="Figure 1. Monthly passenger demand (sum of record values)",
               labels={"Date":"Month", "Passengers":"Passengers"})
fig1.update_layout(hovermode="x unified")
show_and_save(fig1, "figure_01_monthly_passenger_demand.html")
peak_month = monthly_passengers.loc[monthly_passengers["Passengers"].idxmax()]
low_month = monthly_passengers.loc[monthly_passengers["Passengers"].idxmin()]
print(f"Highest month: {peak_month['Date']:%B %Y} ({peak_month['Passengers']:,.0f} passengers). Lowest month: {low_month['Date']:%B %Y} ({low_month['Passengers']:,.0f}).")

Interactive chart saved to: /content/ecomove_outputs/figure_01_monthly_passenger_demand.html
Highest month: March 2024 (2,080,100 passengers). Lowest month: October 2025 (1,409,837).


## 6. Visualisation 2 — Passenger demand by city

**Question:** Which cities account for the largest recorded passenger volumes?  
**Why this chart:** A sorted horizontal bar chart makes category names readable and comparisons easy.  
**Decision use:** This can inform where to review capacity or investment. Total volume alone does not measure service quality and is affected by the scale of each city's operation.

In [40]:
city_summary = df.groupby("City", as_index=False).agg(
    Passengers=("Passengers", "sum"), Records=("Record_ID", "count"),
    Average_On_Time=("On_Time_Percentage", "mean")
).sort_values("Passengers", ascending=False)
fig2 = px.bar(city_summary.sort_values("Passengers"), x="Passengers", y="City", orientation="h",
              hover_data={"Records":True, "Average_On_Time":":.2f"},
              title="Figure 2. Total recorded passenger volume by city",
              labels={"Passengers":"Passengers (sum of records)", "City":"City", "Average_On_Time":"Mean on-time (%)"})
show_and_save(fig2, "figure_02_city_passenger_comparison.html")
print("Highest recorded passenger volume:")
display(city_summary.head(5).round(2))

Interactive chart saved to: /content/ecomove_outputs/figure_02_city_passenger_comparison.html
Highest recorded passenger volume:


,City,Passengers,Records,Average_On_Time
8,Paris,4492481,422,81.96
2,Berlin,4487351,408,81.37
7,Munich,3603328,428,82.01
1,Barcelona,3526320,408,82.80
9,Rome,3418626,420,82.71


## 7. Visualisation 3 — Average on-time performance by country

**Question:** Does average recorded service reliability differ among countries?  
**Why this chart:** A country-level bar chart supports direct comparisons of the mean on-time percentage.  
**Decision use:** Authorities can identify locations for further review. Differences should not be interpreted as a country effect without considering city mix, transport-mode mix, sample size and local operating conditions.

In [41]:
country_summary = df.groupby("Country", as_index=False).agg(
    Average_On_Time=("On_Time_Percentage", "mean"), Average_Delay=("Average_Delay_Minutes", "mean"),
    Records=("Record_ID", "count"), Passengers=("Passengers", "sum")
).sort_values("Average_On_Time", ascending=True)
fig3 = px.bar(country_summary, x="Country", y="Average_On_Time", color="Average_On_Time",
              color_continuous_scale="Blues", text=country_summary["Average_On_Time"].map(lambda x: f"{x:.1f}%"),
              hover_data={"Average_Delay":":.2f", "Records":True, "Passengers":":,.0f"},
              title="Figure 3. Average on-time performance by country",
              labels={"Average_On_Time":"Mean on-time performance (%)", "Average_Delay":"Mean delay (minutes)"})
fig3.update_traces(textposition="outside")
fig3.update_yaxes(range=[0, 100])
show_and_save(fig3, "figure_03_country_reliability.html")
print("Country summary (descriptive averages):")
display(country_summary.round(2))

Interactive chart saved to: /content/ecomove_outputs/figure_03_country_reliability.html
Country summary (descriptive averages):


,Country,Average_On_Time,Average_Delay,Records,Passengers
1,France,81.65,4.46,810,7029009
2,Germany,81.74,4.44,1260,11094843
4,Netherlands,82.03,4.37,880,6062125
0,Austria,82.39,4.32,434,3388470
5,Spain,82.41,4.23,796,6816224
3,Italy,82.47,4.24,820,6519050


## 8. Visualisation 4 — On-time performance by transport mode and route type

**Question:** Which combinations of transport mode and route type have stronger or weaker recorded reliability?  
**Why this chart:** A heatmap makes a two-dimensional set of category averages compact and easy to scan; the cell labels avoid relying on colour alone.  
**Decision use:** Managers can prioritise specific mode/route combinations for investigation. Some combinations may have fewer records, so always check sample counts and avoid assuming small differences are meaningful.

In [42]:
mode_route_counts = df.pivot_table(index="Transport_Mode", columns="Route_Type", values="Record_ID", aggfunc="count")
mode_route_ontime = df.pivot_table(index="Transport_Mode", columns="Route_Type", values="On_Time_Percentage", aggfunc="mean")
fig4 = px.imshow(mode_route_ontime, text_auto=".1f", aspect="auto", color_continuous_scale="Blues",
                 zmin=55, zmax=100, labels={"x":"Route type", "y":"Transport mode", "color":"Mean on-time (%)"},
                 title="Figure 4. Mean on-time percentage by mode and route type")
fig4.update_layout(coloraxis_colorbar={"title":"On-time (%)"})
show_and_save(fig4, "figure_04_mode_route_heatmap.html")
print("Record counts per heatmap cell (review low-count combinations carefully):")
display(mode_route_counts.fillna(0).astype(int))

Interactive chart saved to: /content/ecomove_outputs/figure_04_mode_route_heatmap.html
Record counts per heatmap cell (review low-count combinations carefully):


Route_Type,Airport Connector,Central Route,Commuter Corridor,Suburban Feeder,Tourist Zone,University Area
Transport_Mode,,,,,,
Bike Share,132,126,132,115,140,114
Bus,204,235,222,241,244,229
E-Scooter,84,71,78,81,82,77
Metro,171,180,184,165,170,177
Park and Ride,103,92,96,88,114,92
Tram,124,116,119,127,154,121


## 9. Visualisation 5 — Delay and customer rating relationship

**Question:** Are longer delays associated with different customer ratings in the records?  
**Why this chart:** A scatter plot shows the relationship between two numeric measures and makes clusters and outliers visible. Colour separates transport modes.  
**Decision use:** This can guide further investigation into the customer experience. The data is observational: a relationship does not prove that delay alone caused a rating, and other factors may influence ratings.

In [43]:
fig5 = px.scatter(df, x="Average_Delay_Minutes", y="Customer_Rating", color="Transport_Mode",
                   hover_data=["City", "Country", "Weather", "Event_Day", "On_Time_Percentage", "Passengers"],
                   opacity=0.58, title="Figure 5. Average delay compared with customer rating",
                   labels={"Average_Delay_Minutes":"Average delay (minutes)", "Customer_Rating":"Customer rating (out of 5)"})
show_and_save(fig5, "figure_05_delay_vs_rating.html")
correlation = df[["Average_Delay_Minutes", "Customer_Rating"]].corr().iloc[0, 1]
print(f"Pearson correlation between delay and rating: {correlation:.3f}. This is a descriptive association, not proof of causation.")

Interactive chart saved to: /content/ecomove_outputs/figure_05_delay_vs_rating.html
Pearson correlation between delay and rating: -0.821. This is a descriptive association, not proof of causation.


## 10. Visualisation 6 — Delay distribution by weather

**Question:** How does delay vary across weather categories?  
**Why this chart:** A box plot displays the median, interquartile range and outlying observations, allowing comparison of variability rather than averages alone.  
**Decision use:** Weather conditions with higher or more variable delays may need contingency planning. Category counts differ, and weather may be associated with other conditions, so causality cannot be inferred from this comparison.

In [44]:
weather_order = df.groupby("Weather")["Average_Delay_Minutes"].median().sort_values().index.tolist()
weather_summary = df.groupby("Weather", as_index=False).agg(
    Records=("Record_ID", "count"), Mean_Delay=("Average_Delay_Minutes", "mean"),
    Median_Delay=("Average_Delay_Minutes", "median"), Mean_On_Time=("On_Time_Percentage", "mean"),
    Mean_Rating=("Customer_Rating", "mean")
).sort_values("Mean_Delay", ascending=False)
fig6 = px.box(df, x="Weather", y="Average_Delay_Minutes", color="Weather", points="outliers",
              category_orders={"Weather":weather_order}, title="Figure 6. Distribution of recorded delays by weather",
              labels={"Weather":"Weather condition", "Average_Delay_Minutes":"Average delay (minutes)"})
show_and_save(fig6, "figure_06_delay_distribution_by_weather.html")
display(weather_summary.round(2))

Interactive chart saved to: /content/ecomove_outputs/figure_06_delay_distribution_by_weather.html


,Weather,Records,Mean_Delay,Median_Delay,Mean_On_Time,Mean_Rating
4,Snow,182,8.14,8.45,69.97,3.36
3,Rain,905,5.40,5.50,78.70,3.70
2,Heatwave,243,4.91,5.00,80.19,3.76
5,Windy,306,4.86,5.00,80.42,4.02
1,Cloudy,1293,3.89,4.00,83.46,4.12
0,Clear,2071,3.71,3.70,84.17,4.15


## 11. Visualisation 7 — Event-day versus non-event-day performance

**Question:** Do the records show different demand and reliability on days marked as major event days?  
**Why this chart:** Two aligned panels show delay and on-time percentage separately, avoiding a misleading shared axis for measures with different units.  
**Decision use:** If event days coincide with heavier demand and weaker reliability, managers can consider event-specific capacity and staffing plans. Event records are fewer than non-event records, and the comparison is not a controlled experiment.

In [45]:
event_summary = df.groupby("Event_Day", as_index=False).agg(
    Records=("Record_ID", "count"), Mean_Passengers_Per_Record=("Passengers", "mean"),
    Mean_Delay=("Average_Delay_Minutes", "mean"), Mean_On_Time=("On_Time_Percentage", "mean"),
    Mean_Rating=("Customer_Rating", "mean")
)
event_summary["Event_Day"] = pd.Categorical(event_summary["Event_Day"], categories=["No", "Yes"], ordered=True)
event_summary = event_summary.sort_values("Event_Day")
fig7 = make_subplots(rows=1, cols=2, subplot_titles=("Average delay (minutes)", "Average on-time performance (%)"))
fig7.add_trace(go.Bar(x=event_summary["Event_Day"].astype(str), y=event_summary["Mean_Delay"],
                      marker_color="#0072B2", text=event_summary["Mean_Delay"].map(lambda x:f"{x:.2f}"),
                      textposition="outside", showlegend=False), row=1, col=1)
fig7.add_trace(go.Bar(x=event_summary["Event_Day"].astype(str), y=event_summary["Mean_On_Time"],
                      marker_color="#E69F00", text=event_summary["Mean_On_Time"].map(lambda x:f"{x:.2f}%"),
                      textposition="outside", showlegend=False), row=1, col=2)
fig7.update_layout(title="Figure 7. Reliability on event and non-event days", template="plotly_white", height=440)
fig7.update_xaxes(title="Event day (Yes/No)")
fig7.update_yaxes(title="Average delay (minutes)", row=1, col=1)
fig7.update_yaxes(title="Average on-time (%)", row=1, col=2)
show_and_save(fig7, "figure_07_event_day_performance.html")
display(event_summary.round(2))

Interactive chart saved to: /content/ecomove_outputs/figure_07_event_day_performance.html


,Event_Day,Records,Mean_Passengers_Per_Record,Mean_Delay,Mean_On_Time,Mean_Rating
0,No,4490,7968.86,4.22,82.49,4.02
1,Yes,510,10057.93,5.53,78.21,3.87


## 12. Visualisation 8 — Ticket revenue versus operating costs by transport mode

**Question:** How do the two recorded financial measures compare across modes?  
**Why this chart:** Grouped bars put revenue and operating cost side by side, making the scale and differences visible.  
**Decision use:** Managers can identify modes that merit a closer cost/revenue review. This simple comparison excludes other funding, capital costs, public-service obligations and external benefits, so it is not a complete measure of economic or social value.

In [46]:
mode_finance = df.groupby("Transport_Mode", as_index=False).agg(
    Ticket_Revenue_EUR=("Ticket_Revenue_EUR", "sum"), Operating_Cost_EUR=("Operating_Cost_EUR", "sum"),
    Passengers=("Passengers", "sum")
)
mode_finance["Revenue_Minus_Cost_EUR"] = mode_finance["Ticket_Revenue_EUR"] - mode_finance["Operating_Cost_EUR"]
finance_long = mode_finance.melt(id_vars="Transport_Mode", value_vars=["Ticket_Revenue_EUR", "Operating_Cost_EUR"],
                                var_name="Measure", value_name="EUR")
finance_long["Measure"] = finance_long["Measure"].map({"Ticket_Revenue_EUR":"Ticket revenue", "Operating_Cost_EUR":"Operating cost"})
fig8 = px.bar(finance_long, x="Transport_Mode", y="EUR", color="Measure", barmode="group",
              title="Figure 8. Ticket revenue and operating cost by transport mode",
              labels={"Transport_Mode":"Transport mode", "EUR":"Total EUR", "Measure":"Measure"})
show_and_save(fig8, "figure_08_revenue_vs_cost.html")
print("Revenue less operating cost by mode (not full accounting profit):")
display(mode_finance[["Transport_Mode", "Ticket_Revenue_EUR", "Operating_Cost_EUR", "Revenue_Minus_Cost_EUR"]].sort_values("Revenue_Minus_Cost_EUR").round(2))

Interactive chart saved to: /content/ecomove_outputs/figure_08_revenue_vs_cost.html
Revenue less operating cost by mode (not full accounting profit):


,Transport_Mode,Ticket_Revenue_EUR,Operating_Cost_EUR,Revenue_Minus_Cost_EUR
0,Bike Share,310355.03,5912287.24,-5601932.21
2,E-Scooter,261278.13,3536959.56,-3275681.43
1,Bus,12002118.92,11992457.44,9661.48
4,Park and Ride,10066523.80,8283138.05,1783385.75
5,Tram,15666167.52,13355946.79,2310220.73
3,Metro,73334170.86,69029226.89,4304943.97


## 13. Visualisation 9 — Accessibility complaint rate by transport mode

**Question:** Which modes have the highest recorded accessibility-complaint rate relative to passenger volume?  
**Why this chart:** A rate per 10,000 passengers makes comparisons fairer than ranking only the raw complaint count, which tends to be higher where passenger volumes are larger.  
**Decision use:** This can help prioritise accessibility audits and passenger engagement. Complaint counts depend on reporting behaviour, and rates for small passenger volumes can be unstable; they should be investigated alongside severity and context.

In [47]:
accessibility_summary = df.groupby("Transport_Mode", as_index=False).agg(
    Passengers=("Passengers", "sum"), Complaints=("Accessibility_Complaints", "sum"),
    Records=("Record_ID", "count")
)
accessibility_summary["Complaints_per_10k_Passengers"] = (
    accessibility_summary["Complaints"] / accessibility_summary["Passengers"].replace(0, np.nan) * 10000
)
accessibility_summary = accessibility_summary.sort_values("Complaints_per_10k_Passengers", ascending=True)
fig9 = px.bar(accessibility_summary, x="Complaints_per_10k_Passengers", y="Transport_Mode", orientation="h",
              text=accessibility_summary["Complaints_per_10k_Passengers"].map(lambda x:f"{x:.1f}"),
              hover_data={"Complaints":True, "Passengers":":,.0f", "Records":True},
              title="Figure 9. Accessibility complaints per 10,000 recorded passengers",
              labels={"Complaints_per_10k_Passengers":"Complaints per 10,000 passengers", "Transport_Mode":"Transport mode"})
fig9.update_traces(textposition="outside")
fig9.update_layout(yaxis={"categoryorder":"total ascending"})
show_and_save(fig9, "figure_09_accessibility_complaint_rate.html")
display(accessibility_summary.round(3))

Interactive chart saved to: /content/ecomove_outputs/figure_09_accessibility_complaint_rate.html


,Transport_Mode,Passengers,Complaints,Records,Complaints_per_10k_Passengers
3,Metro,25757522,38240,1047,14.846
5,Tram,6380878,9621,761,15.078
4,Park and Ride,3265220,5183,585,15.873
0,Bike Share,268765,457,759,17.004
1,Bus,5073394,8859,1375,17.462
2,E-Scooter,163942,301,473,18.360


## 14. Visualisation 10 — Environmental contribution by mode

**Question:** How are the dataset's recorded CO₂ savings distributed across transport modes?  
**Why this chart:** A treemap communicates each mode's contribution to a total; colour also encodes recorded CO₂ savings per passenger to provide a different perspective from total volume.  
**Decision use:** It supports sustainability discussions alongside demand, energy use, cost and service coverage. CO₂ saved is a supplied estimate whose calculation method is not documented in this file, so avoid treating it as independently verified emissions accounting.

In [48]:
environment_summary = df.groupby("Transport_Mode", as_index=False).agg(
    CO2_Saved_KG=("CO2_Saved_KG", "sum"), Energy_Consumption_KWh=("Energy_Consumption_KWh", "sum"),
    Passengers=("Passengers", "sum")
)
environment_summary["CO2_Saved_per_Passenger"] = environment_summary["CO2_Saved_KG"] / environment_summary["Passengers"].replace(0, np.nan)
environment_summary["Energy_kWh_per_1000_Passengers"] = environment_summary["Energy_Consumption_KWh"] / environment_summary["Passengers"].replace(0, np.nan) * 1000
fig10 = px.treemap(environment_summary, path=["Transport_Mode"], values="CO2_Saved_KG",
                   color="CO2_Saved_per_Passenger", color_continuous_scale="Greens",
                   hover_data={"CO2_Saved_KG":":,.0f", "CO2_Saved_per_Passenger":":.3f",
                               "Passengers":":,.0f", "Energy_Consumption_KWh":":,.0f"},
                   title="Figure 10. Total recorded CO₂ savings by transport mode (colour = kg per passenger)",
                   labels={"CO2_Saved_KG":"CO₂ saved (kg)", "CO2_Saved_per_Passenger":"CO₂ saved per passenger (kg)"})
show_and_save(fig10, "figure_10_environmental_contribution.html")
display(environment_summary.round(3))

Interactive chart saved to: /content/ecomove_outputs/figure_10_environmental_contribution.html


,Transport_Mode,CO2_Saved_KG,Energy_Consumption_KWh,Passengers,CO2_Saved_per_Passenger,Energy_kWh_per_1000_Passengers
0,Bike Share,49132.7,0.0,268765,0.183,0.000
1,Bus,414163.1,386822.2,5073394,0.082,76.245
2,E-Scooter,21651.3,3004.1,163942,0.132,18.324
3,Metro,3150917.3,1439059.8,25757522,0.122,55.869
4,Park and Ride,199191.2,115959.5,3265220,0.061,35.514
5,Tram,646918.7,389196.2,6380878,0.101,60.994


## 15. Combined findings and evidence-led recommendations

The following section calculates the principal comparisons directly from the loaded data. These statements are descriptive, not causal. Use them as an evidence base for the critical report, and refer to figure numbers when discussing the findings.

In [49]:
# Verify the delay-versus-customer-rating direction reported in Word Figure 5.
delay_rating_r = df["Average_Delay_Minutes"].corr(df["Customer_Rating"], method="pearson")
print(f"Pearson r (average delay vs customer rating): {delay_rating_r:.3f}. "
      "A negative coefficient indicates higher delays are associated with lower ratings; this is not causal evidence.")

# Key data-driven findings: computed from this workbook each time the notebook runs.
weakest_weather = weather_summary.sort_values("Mean_On_Time").iloc[0]
best_weather = weather_summary.sort_values("Mean_On_Time", ascending=False).iloc[0]
mode_summary = df.groupby("Transport_Mode", as_index=False).agg(
    Records=("Record_ID", "count"), Passengers=("Passengers", "sum"),
    Mean_Delay=("Average_Delay_Minutes", "mean"), Mean_On_Time=("On_Time_Percentage", "mean"),
    Mean_Rating=("Customer_Rating", "mean"), Revenue=("Ticket_Revenue_EUR", "sum"),
    Cost=("Operating_Cost_EUR", "sum"), CO2_Saved=("CO2_Saved_KG", "sum"),
    Energy=("Energy_Consumption_KWh", "sum"), Complaints=("Accessibility_Complaints", "sum")
)
mode_summary["Revenue_Minus_Cost"] = mode_summary["Revenue"] - mode_summary["Cost"]
mode_summary["Complaints_per_10k_Passengers"] = mode_summary["Complaints"] / mode_summary["Passengers"].replace(0, np.nan) * 10000
worst_reliability_mode = mode_summary.sort_values("Mean_On_Time").iloc[0]
best_reliability_mode = mode_summary.sort_values("Mean_On_Time", ascending=False).iloc[0]
city_top = city_summary.sort_values("Passengers", ascending=False).iloc[0]
event_no = event_summary[event_summary["Event_Day"].astype(str) == "No"].iloc[0]
event_yes = event_summary[event_summary["Event_Day"].astype(str) == "Yes"].iloc[0]
financial_gap = df["Ticket_Revenue_EUR"].sum() - df["Operating_Cost_EUR"].sum()

findings = [
    f"**Dataset coverage:** {len(df):,} records from {df['Date'].min():%d %B %Y} to {df['Date'].max():%d %B %Y}; {df['City'].nunique()} cities in {df['Country'].nunique()} countries.",
    f"**Demand:** {city_top['City']} has the highest total recorded passenger volume ({city_top['Passengers']:,.0f}) among the cities in this dataset.",
    f"**Reliability by mode:** {worst_reliability_mode['Transport_Mode']} has the lowest mean on-time performance ({worst_reliability_mode['Mean_On_Time']:.2f}%), while {best_reliability_mode['Transport_Mode']} has the highest ({best_reliability_mode['Mean_On_Time']:.2f}%). These are record averages and may reflect differences in route mix and operating conditions.",
    f"**Weather:** {weakest_weather['Weather']} has the lowest mean on-time performance ({weakest_weather['Mean_On_Time']:.2f}%) and {weakest_weather['Mean_Delay']:.2f} minutes average delay among weather categories. {best_weather['Weather']} has the highest mean on-time performance ({best_weather['Mean_On_Time']:.2f}%).",
    f"**Event days:** mean delay is {event_yes['Mean_Delay']:.2f} minutes on event-day records compared with {event_no['Mean_Delay']:.2f} minutes on non-event records; average on-time performance is {event_yes['Mean_On_Time']:.2f}% vs {event_no['Mean_On_Time']:.2f}%.",
    f"**Financial indicator:** total ticket revenue minus operating costs is €{financial_gap:,.2f} across the records. This is a simple two-column comparison, not full accounting profit.",
    f"**Sustainability and access:** the workbook records {df['CO2_Saved_KG'].sum():,.0f} kg of CO₂ saved and {df['Accessibility_Complaints'].sum():,} accessibility complaints; compare modes using both totals and passenger-normalised rates."
]
for item in findings:
    display(Markdown("- " + item))

recommendations = pd.DataFrame([
    ["Reliability", "Investigate lower-performing modes and the route/mode cells with weaker on-time results; review traffic, timetable, vehicle and staffing factors before prescribing a fix."],
    ["Weather preparedness", "Review contingency procedures for adverse weather and compare performance at similar routes and times before attributing delays to weather."],
    ["Event planning", "Use the observed event-day demand/reliability differences to plan extra capacity, staffing and passenger communications, then monitor whether these actions improve results."],
    ["Financial management", "Review revenue and costs by mode with a full cost-allocation method and public-service objectives; do not make service cuts using revenue minus cost alone."],
    ["Accessibility", "Investigate modes with higher complaint rates, examine complaint types and severity, and engage affected passengers; check both absolute counts and rates."],
    ["Sustainability", "Interpret CO₂ savings alongside passenger volumes, energy consumption and the methodology behind the CO₂ estimate; do not compare modes by total CO₂ alone."],
    ["Dashboard governance", "Refresh data regularly, define each KPI clearly, check source quality and use readable labels/colour choices. Treat correlations as prompts for investigation rather than proof of cause."]
], columns=["Area", "Evidence-based recommendation"])
display(recommendations)

Pearson r (average delay vs customer rating): -0.821. A negative coefficient indicates higher delays are associated with lower ratings; this is not causal evidence.


- **Dataset coverage:** 5,000 records from 01 January 2024 to 31 December 2025; 12 cities in 6 countries.

- **Demand:** Paris has the highest total recorded passenger volume (4,492,481) among the cities in this dataset.

- **Reliability by mode:** Bus has the lowest mean on-time performance (74.92%), while Bike Share has the highest (90.85%). These are record averages and may reflect differences in route mix and operating conditions.

- **Weather:** Snow has the lowest mean on-time performance (69.97%) and 8.14 minutes average delay among weather categories. Clear has the highest mean on-time performance (84.17%).

- **Event days:** mean delay is 5.53 minutes on event-day records compared with 4.22 minutes on non-event records; average on-time performance is 78.21% vs 82.49%.

- **Financial indicator:** total ticket revenue minus operating costs is €-469,401.71 across the records. This is a simple two-column comparison, not full accounting profit.

- **Sustainability and access:** the workbook records 4,481,974 kg of CO₂ saved and 62,661 accessibility complaints; compare modes using both totals and passenger-normalised rates.

,Area,Evidence-based recommendation
0,Reliability,Investigate lower-performing modes and the rou...
1,Weather preparedness,Review contingency procedures for adverse weat...
2,Event planning,Use the observed event-day demand/reliability ...
3,Financial management,Review revenue and costs by mode with a full c...
4,Accessibility,"Investigate modes with higher complaint rates,..."
5,Sustainability,Interpret CO₂ savings alongside passenger volu...
6,Dashboard governance,"Refresh data regularly, define each KPI clearl..."


## 16. Interactive dashboard — Streamlit application

The brief requires an interactive dashboard. The following cell writes a companion `ecomove_dashboard.py` file. The application includes filters for date range, city, country, transport mode, route type, weather and event day; it recalculates key indicators and charts from the filtered data.

**To run it:** install the required libraries (`pip install streamlit pandas plotly openpyxl`), keep the dashboard script and supplied Excel file in the same folder, then run `streamlit run ecomove_dashboard.py` in a terminal. If Streamlit is not installed in your current environment, the analysis cells in this notebook still work; run the dashboard in a Python environment with Streamlit installed.

In [50]:
# Complete Streamlit source: corresponds to original notebook code cell 32.
# Include the whole application source (not only this generator cell) in the Word appendix.
dashboard_source = '''from pathlib import Path
import io
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
from plotly.subplots import make_subplots

st.set_page_config(page_title="EcoMove Mobility Insights", page_icon="🚌", layout="wide")
st.title("EcoMove Mobility Insights Dashboard")
st.caption("COM7021 Data Visualisation | Evidence-led exploration of demand, reliability, finance, accessibility and sustainability")

# Use the original dataset filename.
DATA_FILENAMES = [
    "[5442] EcoMoveMobility.xlsx"
]
data_path = next(
    (Path(__file__).resolve().parent / name for name in DATA_FILENAMES
     if (Path(__file__).resolve().parent / name).is_file()),
    None,
)

@st.cache_data
def load_data(file_bytes=None):
    if file_bytes is not None:
        source = io.BytesIO(file_bytes)
        data = pd.read_excel(source, sheet_name="Dataset")
    else:
        data = pd.read_excel(data_path, sheet_name="Dataset")
    if pd.api.types.is_numeric_dtype(data["Date"]):
        data["Date"] = pd.to_datetime(data["Date"], unit="D", origin="1899-12-30", errors="coerce")
    else:
        data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
    numeric_cols = ["Passengers", "Trips", "Average_Delay_Minutes", "On_Time_Percentage",
                    "Ticket_Revenue_EUR", "Operating_Cost_EUR", "CO2_Saved_KG",
                    "Energy_Consumption_KWh", "Customer_Rating", "Accessibility_Complaints"]
    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    return data

if data_path is None:
    uploaded = st.sidebar.file_uploader("Upload the supplied EcoMoveMobility Excel file", type=["xlsx"])
    if uploaded is None:
        st.info("Place the supplied EcoMoveMobility Excel file alongside the dashboard, or upload it using the sidebar.")
        st.stop()
    raw = uploaded.getvalue()
    df = load_data(raw)
else:
    df = load_data()

required_cols = ["Date", "City", "Country", "Transport_Mode", "Route_Type", "Weather", "Event_Day",
                 "Passengers", "Trips", "Average_Delay_Minutes", "On_Time_Percentage",
                 "Ticket_Revenue_EUR", "Operating_Cost_EUR", "CO2_Saved_KG", "Energy_Consumption_KWh",
                 "Customer_Rating", "Accessibility_Complaints"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    st.error("The workbook is missing required columns: " + ", ".join(missing_cols))
    st.stop()
df = df.dropna(subset=["Date"]).copy()

st.sidebar.header("Filter the data")
min_date, max_date = df["Date"].min().date(), df["Date"].max().date()
date_range = st.sidebar.date_input("Date range", value=(min_date, max_date), min_value=min_date, max_value=max_date)

def category_filter(label, column):
    options = sorted(df[column].dropna().astype(str).unique().tolist())
    return st.sidebar.multiselect(label, ["All"] + options, default=["All"])

cities = category_filter("City", "City")
countries = category_filter("Country", "Country")
modes = category_filter("Transport mode", "Transport_Mode")
routes = category_filter("Route type", "Route_Type")
weather = category_filter("Weather", "Weather")
events = category_filter("Event day", "Event_Day")

filtered = df.copy()
if isinstance(date_range, (tuple, list)) and len(date_range) == 2:
    filtered = filtered[filtered["Date"].dt.date.between(date_range[0], date_range[1])]
for col, selected in [("City", cities), ("Country", countries), ("Transport_Mode", modes),
                      ("Route_Type", routes), ("Weather", weather), ("Event_Day", events)]:
    if selected and "All" not in selected:
        filtered = filtered[filtered[col].astype(str).isin(selected)]

if filtered.empty:
    st.warning("No records match the selected filters. Choose a wider date range or add categories.")
    st.stop()

n_passengers = filtered["Passengers"].sum()
mean_ontime = filtered["On_Time_Percentage"].mean()
mean_delay = filtered["Average_Delay_Minutes"].mean()
net_revenue = filtered["Ticket_Revenue_EUR"].sum() - filtered["Operating_Cost_EUR"].sum()
co2 = filtered["CO2_Saved_KG"].sum()

k1, k2, k3, k4, k5 = st.columns(5)
k1.metric("Passengers (sum of records)", f"{n_passengers:,.0f}")
k2.metric("Average on-time", f"{mean_ontime:.1f}%")
k3.metric("Average delay", f"{mean_delay:.2f} min")
k4.metric("Revenue − operating cost", f"€{net_revenue:,.0f}")
k5.metric("CO₂ saved", f"{co2:,.0f} kg")
st.caption(f"Showing {len(filtered):,} of {len(df):,} records. KPI values recalculate when filters change.")

monthly = filtered.set_index("Date").resample("MS")["Passengers"].sum().reset_index()
city_summary = filtered.groupby("City", as_index=False)["Passengers"].sum().sort_values("Passengers", ascending=True)
fig1 = px.line(monthly, x="Date", y="Passengers", markers=True, title="Monthly passenger demand", template="plotly_white")
fig1.update_yaxes(title="Passengers (sum of records)")
fig2 = px.bar(city_summary, x="Passengers", y="City", orientation="h", title="Passenger demand by city", template="plotly_white")
fig2.update_layout(yaxis={"categoryorder":"total ascending"})
finance = filtered.groupby("Transport_Mode", as_index=False)[["Ticket_Revenue_EUR", "Operating_Cost_EUR"]].sum()
finance_long = finance.melt(id_vars="Transport_Mode", var_name="Measure", value_name="EUR")
finance_long["Measure"] = finance_long["Measure"].map({"Ticket_Revenue_EUR":"Ticket revenue", "Operating_Cost_EUR":"Operating cost"})
fig3 = px.bar(finance_long, x="Transport_Mode", y="EUR", color="Measure", barmode="group", title="Ticket revenue and operating costs by mode", template="plotly_white")
fig3.update_yaxes(title="EUR")
fig4 = px.scatter(filtered, x="Average_Delay_Minutes", y="Customer_Rating", color="Transport_Mode", hover_data=["City", "Country", "Weather", "Event_Day", "On_Time_Percentage"], opacity=0.62, title="Delay and customer rating (record level)", template="plotly_white")
fig4.update_xaxes(title="Average delay (minutes)")
fig4.update_yaxes(title="Customer rating (out of 5)")
weather_order = filtered.groupby("Weather")["Average_Delay_Minutes"].median().sort_values().index.tolist()
fig5 = px.box(filtered, x="Weather", y="Average_Delay_Minutes", color="Weather", category_orders={"Weather":weather_order}, points="outliers", title="Delay distribution by weather", template="plotly_white")
fig5.update_yaxes(title="Average delay (minutes)")
heat = filtered.pivot_table(index="Transport_Mode", columns="Route_Type", values="On_Time_Percentage", aggfunc="mean")
fig6 = px.imshow(heat, text_auto=".1f", aspect="auto", color_continuous_scale="Blues", zmin=55, zmax=100, labels={"x":"Route type", "y":"Transport mode", "color":"On-time (%)"}, title="Average on-time performance by mode and route type", template="plotly_white")
event_summary = filtered.groupby("Event_Day", as_index=False).agg(Average_Delay_Minutes=("Average_Delay_Minutes", "mean"), On_Time_Percentage=("On_Time_Percentage", "mean"), Passengers_Per_Record=("Passengers", "mean"))
event_summary["Event_Day"] = pd.Categorical(event_summary["Event_Day"], categories=["No", "Yes"], ordered=True)
event_summary = event_summary.sort_values("Event_Day")
fig7 = make_subplots(rows=1, cols=2, subplot_titles=("Average delay (minutes)", "Average on-time performance (%)"))
for col, metric, color in [(1, "Average_Delay_Minutes", "#0072B2"), (2, "On_Time_Percentage", "#E69F00")]:
    fig7.add_trace(go.Bar(x=event_summary["Event_Day"].astype(str), y=event_summary[metric], marker_color=color, showlegend=False, text=event_summary[metric].round(2), textposition="outside"), row=1, col=col)
fig7.update_layout(title="Service performance on event and non-event days", template="plotly_white", height=420)
fig7.update_xaxes(title="Event day")
fig7.update_yaxes(title="Minutes", row=1, col=1)
fig7.update_yaxes(title="Percent", row=1, col=2)
complaints = filtered.groupby("Transport_Mode", as_index=False).agg(Passengers=("Passengers", "sum"), Accessibility_Complaints=("Accessibility_Complaints", "sum"))
complaints["Complaints_per_10k_Passengers"] = complaints["Accessibility_Complaints"] / complaints["Passengers"].replace(0, pd.NA) * 10000
complaints = complaints.sort_values("Complaints_per_10k_Passengers", ascending=True)
fig8 = px.bar(complaints, x="Complaints_per_10k_Passengers", y="Transport_Mode", orientation="h", text=complaints["Complaints_per_10k_Passengers"].map(lambda x: f"{x:.1f}"), title="Accessibility complaints per 10,000 passengers", template="plotly_white")
fig8.update_layout(yaxis={"categoryorder":"total ascending"})
fig8.update_xaxes(title="Complaints per 10,000 passengers")
carbon = filtered.groupby("Transport_Mode", as_index=False).agg(CO2_Saved_KG=("CO2_Saved_KG", "sum"), Passengers=("Passengers", "sum"))
carbon["CO2_Saved_per_Passenger"] = carbon["CO2_Saved_KG"] / carbon["Passengers"].replace(0, pd.NA)
fig9 = px.treemap(carbon, path=["Transport_Mode"], values="CO2_Saved_KG", color="CO2_Saved_per_Passenger", color_continuous_scale="Blues", hover_data={"CO2_Saved_KG":":,.0f", "CO2_Saved_per_Passenger":":.3f", "Passengers":":,.0f"}, title="Recorded CO₂ savings by transport mode", template="plotly_white")

left, right = st.columns(2)
with left:
    st.plotly_chart(fig1, use_container_width=True)
    st.plotly_chart(fig3, use_container_width=True)
    st.plotly_chart(fig5, use_container_width=True)
    st.plotly_chart(fig8, use_container_width=True)
with right:
    st.plotly_chart(fig2, use_container_width=True)
    st.plotly_chart(fig4, use_container_width=True)
    st.plotly_chart(fig6, use_container_width=True)
    st.plotly_chart(fig7, use_container_width=True)
st.plotly_chart(fig9, use_container_width=True)

with st.expander("How to interpret this dashboard and its limitations"):
    st.markdown("""
    - Values are derived from the provided dataset and change with the filters.
    - 'Passengers' is the sum of the supplied record-level passenger counts; it should not be interpreted as unique people unless the data documentation confirms that.
    - A relationship between delays, weather, events and ratings does not prove causation.
    - Complaint rates use accessibility complaints divided by the sum of passengers, multiplied by 10,000. Rates based on small volumes should be interpreted cautiously.
    - Revenue minus operating cost is a simple comparison of the two supplied fields, not a complete accounting profit measure.
    """)
'''
app_path = Path.cwd() / 'ecomove_dashboard.py'
app_path.write_text(dashboard_source, encoding='utf-8')
print(f'Streamlit dashboard source created: {app_path.resolve()}')
print('Run with: streamlit run ecomove_dashboard.py')


Streamlit dashboard source created: /content/ecomove_dashboard.py
Run with: streamlit run ecomove_dashboard.py


## 17. Critical evaluation and limitations (for Task 2)

- **Audience and design:** Sorted bars support rapid comparisons; time-series charts show temporal patterns; box plots show spread; scatter plots explore associations; and heatmaps reveal combinations that may need attention. Clear titles, units, readable labels, accessible colour choices and numerical labels are important for both technical and non-technical audiences.
- **Data quality:** This notebook checks completeness, duplicate records and date parsing. Those checks do not validate how the original passenger, delay, cost or CO₂ measurements were collected. No missing cells does not guarantee accurate data.
- **Aggregation:** The passenger column is interpreted as the recorded passenger volume per row. Average performance values are unweighted record means. A passenger-weighted average could give a different result and should be considered if the business question is passenger experience.
- **Causality:** The dataset is observational. Weather and event-day differences may be confounded by route, city, mode or time. The charts show associations and descriptive differences, not proven causal effects.
- **Accessibility:** Complaint rates are normalised per 10,000 recorded passengers, but reporting propensities, severity and access to complaint channels may differ. Rate and count should be considered together.
- **Finance and sustainability:** Revenue less operating cost is not full accounting profit and excludes other benefits/costs unless the source fields incorporate them. The calculation method for CO₂ saved is not described in the workbook, so those values should be treated as provided estimates.
- **Dashboard effectiveness:** Filters let managers move from an overall pattern to a city, mode or weather condition. However, the dashboard depends on timely, accurate input data, and too many filters or unclear metric definitions can confuse non-technical users. KPI definitions and caveats should remain visible.

For LO4, connect these findings to evidence from academic and professional sources on data visualisation, transport planning, accessibility and sustainable mobility. Apply the required Harvard referencing style in the Word report, and ensure external claims are supported by sources rather than by the dataset alone.

## 18. Reproducibility, file outputs and final checks

This cell summarises the outputs and checks that the ten chart exports and dashboard source were generated. The HTML files can be opened in a browser; the `.py` file is the dashboard application source. The original workbook is not modified. **The notebook accepts the uploaded dataset filename and the original shorter name. Run all cells to refresh all outputs before submission.**

In [51]:
# Final checks and artifact list
assert len(df) > 0, "The input dataset is empty."
assert not missing_columns, f"Missing required columns: {missing_columns}"
assert df["Date"].notna().all(), "Some dates failed parsing; investigate before submission."
expected_chart_files = [
    'figure_01_monthly_passenger_demand.html',
    'figure_02_city_passenger_comparison.html',
    'figure_03_country_reliability.html',
    'figure_04_mode_route_heatmap.html',
    'figure_05_delay_vs_rating.html',
    'figure_06_delay_distribution_by_weather.html',
    'figure_07_event_day_performance.html',
    'figure_08_revenue_vs_cost.html',
    'figure_09_accessibility_complaint_rate.html',
    'figure_10_environmental_contribution.html',
]
assert all((OUTPUT_DIR / name).exists() for name in expected_chart_files), \
    "One or more of the ten required chart HTML exports is missing."
assert (Path.cwd() / "ecomove_dashboard.py").exists(), "Dashboard source file was not generated."

print("Final validation summary")
print(f"Rows: {len(df):,}; columns: {df.shape[1]}; missing cells: {int(df.isna().sum().sum()):,}; exact duplicate rows: {int(df.duplicated().sum()):,}")
print(f"Chart HTML files exported: {len(expected_chart_files)}")
print(f"Dashboard source: {(Path.cwd() / 'ecomove_dashboard.py').resolve()}")
print("Reminder: add screenshots and the Python code to the Word report as the assessment brief instructs; include student ID on the submitted file.")

# Reconciliation with the existing COM7021 Word report: Table 1, Table 2 and Figure 5.
# The checks below compare recorded summary values; they do not validate the real-world
# accuracy of the dataset, nor support causal conclusions about weather or events.
report_benchmarks = {
    "Records": (len(df), 5000, 0),
    "Columns": (df.shape[1], 18, 0),
    "Passenger total": (df["Passengers"].sum(), 40909721, 0),
    "Trip total": (df["Trips"].sum(), 608862, 0),
    "Average on-time (%)": (df["On_Time_Percentage"].mean(), 82.06, 0.0051),
    "Average delay (minutes)": (df["Average_Delay_Minutes"].mean(), 4.35, 0.0051),
    "Average customer rating": (df["Customer_Rating"].mean(), 4.01, 0.0051),
    "Ticket revenue (EUR)": (df["Ticket_Revenue_EUR"].sum(), 111640614.26, 0.011),
    "Operating cost (EUR)": (df["Operating_Cost_EUR"].sum(), 112110015.97, 0.011),
    "Revenue less cost (EUR)": (df["Ticket_Revenue_EUR"].sum()-df["Operating_Cost_EUR"].sum(), -469401.71, 0.011),
    "CO2 saved (kg)": (df["CO2_Saved_KG"].sum(), 4481974.30, 0.011),
    "Accessibility complaints": (df["Accessibility_Complaints"].sum(), 62661, 0),
    "Delay–rating Pearson r": (df["Average_Delay_Minutes"].corr(df["Customer_Rating"]), -0.821, 0.00051),
}
reconciliation = pd.DataFrame([
    {"Measure": measure, "Calculated": float(actual), "Word report": float(expected),
     "Match": abs(float(actual)-float(expected)) <= tolerance}
    for measure,(actual,expected,tolerance) in report_benchmarks.items()
])
display(reconciliation.round(5))
assert reconciliation["Match"].all(), "A reported numerical result does not match the uploaded dataset: inspect the reconciliation table."
print("PASS: All 13 listed report results agree with the supplied workbook within the displayed rounding.")
print("Interpretation: a negative delay–rating coefficient is an association, not evidence of causality.")


Final validation summary
Rows: 5,000; columns: 18; missing cells: 0; exact duplicate rows: 0
Chart HTML files exported: 10
Dashboard source: /content/ecomove_dashboard.py
Reminder: add screenshots and the Python code to the Word report as the assessment brief instructs; include student ID on the submitted file.


,Measure,Calculated,Word report,Match
0,Records,5.000000e+03,5.000000e+03,True
1,Columns,1.800000e+01,1.800000e+01,True
2,Passenger total,4.090972e+07,4.090972e+07,True
3,Trip total,6.088620e+05,6.088620e+05,True
4,Average on-time (%),8.205736e+01,8.206000e+01,True
5,Average delay (minutes),4.353320e+00,4.350000e+00,True
6,Average customer rating,4.008400e+00,4.010000e+00,True
7,Ticket revenue (EUR),1.116406e+08,1.116406e+08,True
8,Operating cost (EUR),1.121100e+08,1.121100e+08,True
9,Revenue less cost (EUR),-4.694017e+05,-4.694017e+05,True


PASS: All 13 listed report results agree with the supplied workbook within the displayed rounding.
Interpretation: a negative delay–rating coefficient is an association, not evidence of causality.
